# Notebook 05: Final Preprocessing and Dataset Split

This notebook creates the final modelling-ready dataset for the FIA tree carbon storage prediction project.

The input is the cleaned live-tree dataset created in Notebook 03. This notebook selects the final predictors, removes leakage-risk variables, creates simple engineered features, and splits the data into training, validation and test sets for the group’s modelling notebooks.

This notebook does not train machine learning models. Encoding, scaling and imputation should be performed later inside each modelling notebook using train-data-only preprocessing pipelines.

## 1. Import libraries and project paths

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.paths import INTERIM_DIR, PROCESSED_DIR, OUTPUTS_DIR, create_project_dirs
from src.fia_preprocessing import (
    load_cleaned_dataset,
    create_engineered_features,
    select_final_model_columns,
    create_stratified_regression_split,
    create_final_data_dictionary,
    save_preprocessing_outputs
)

create_project_dirs()

print("Project root:", PROJECT_ROOT)
print("Interim folder:", INTERIM_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Outputs folder:", OUTPUTS_DIR)

Project root: e:\BA_DV_Project\COMM074_FIA_Project
Interim folder: E:\BA_DV_Project\COMM074_FIA_Project\data\interim
Processed folder: E:\BA_DV_Project\COMM074_FIA_Project\data\processed
Outputs folder: E:\BA_DV_Project\COMM074_FIA_Project\outputs


## 2. Check cleaned input dataset

In [2]:
cleaned_path = INTERIM_DIR / "fia_tree_carbon_clean_unencoded.parquet"

print("Cleaned dataset exists:", cleaned_path.exists())
print("Cleaned dataset path:", cleaned_path)

Cleaned dataset exists: True
Cleaned dataset path: E:\BA_DV_Project\COMM074_FIA_Project\data\interim\fia_tree_carbon_clean_unencoded.parquet


## 3. Load cleaned dataset

In [3]:
df = load_cleaned_dataset()

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (4540549, 55)


,CN,PLT_CN,PREV_TRE_CN,INVYR,STATECD,UNITCD,COUNTYCD,PLOT,SUBP,TREE,...,FLDTYPCD,OWNGRPCD,RESERVCD,SITECLCD,STDAGE,STDSZCD,BALIVE,ALSTK,GSSTK,has_cond_match
0,157825670010854,157531323010854,NaN,1982,1,5,111,90055,104,3,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
1,157825671010854,157531323010854,NaN,1982,1,5,111,90055,104,4,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
2,157825672010854,157531323010854,NaN,1982,1,5,111,90055,104,5,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
3,157825673010854,157531323010854,NaN,1982,1,5,111,90055,104,6,...,NaN,40.0,0.0,5.0,10.0,3.0,NaN,142.921,102.644,True
4,157825692010854,157531325010854,NaN,1982,1,5,111,90057,101,1,...,NaN,40.0,0.0,4.0,30.0,1.0,NaN,121.504,82.939,True


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4540549 entries, 0 to 4540548
Data columns (total 55 columns):
 #   Column          Dtype  
---  ------          -----  
 0   CN              int64  
 1   PLT_CN          int64  
 2   PREV_TRE_CN     str    
 3   INVYR           int64  
 4   STATECD         int64  
 5   UNITCD          int64  
 6   COUNTYCD        int64  
 7   PLOT            int64  
 8   SUBP            int64  
 9   TREE            int64  
 10  CONDID          int64  
 11  STATUSCD        int64  
 12  SPCD            int64  
 13  DIA             float64
 14  DIAHTCD         float64
 15  HT              float64
 16  HTCD            str    
 17  ACTUALHT        float64
 18  CR              float64
 19  CCLCD           float64
 20  CARBON_AG       float64
 21  CARBON_BG       float64
 22  DRYBIO_AG       float64
 23  DRYBIO_BG       float64
 24  VOLCFNET        float64
 25  VOLCSNET        float64
 26  VOLBFNET        float64
 27  TPA_UNADJ       float64
 28  PLOT_CN_KEY     int64  

## 4. Validate essential columns

In [5]:
essential_cols = [
    "CARBON_AG",
    "DIA",
    "SPCD",
    "STATECD",
    "FORTYPCD"
]

missing_essential = [c for c in essential_cols if c not in df.columns]

print("Missing essential columns:", missing_essential)

Missing essential columns: []


## 5. Create engineered features

In [6]:
df_engineered = create_engineered_features(df)

engineered_cols = [
    "DIA_squared",
    "DIA_HT_interaction",
    "CARBON_AG_log",
    "diameter_class"
]

[c for c in engineered_cols if c in df_engineered.columns]

Engineered features created.


['DIA_squared', 'DIA_HT_interaction', 'CARBON_AG_log', 'diameter_class']

In [7]:
df_engineered[["DIA", "HT", "DIA_squared", "DIA_HT_interaction", "CARBON_AG", "CARBON_AG_log", "diameter_class"]].head()

,DIA,HT,DIA_squared,DIA_HT_interaction,CARBON_AG,CARBON_AG_log,diameter_class
0,2.6,NaN,6.76,NaN,5.930856,1.935983,0-5
1,2.3,NaN,5.29,NaN,4.353856,1.677817,0-5
2,2.6,NaN,6.76,NaN,5.930856,1.935983,0-5
3,2.2,NaN,4.84,NaN,3.889810,1.587153,0-5
4,9.7,68.0,94.09,659.6,240.832853,5.488247,5-10


The engineered features follow the EDA findings from Notebook 04. `DIA_squared` and `DIA_HT_interaction` are included because the relationship between diameter, height and carbon storage was clearly non-linear. `CARBON_AG_log` is included as an optional transformed target because the raw carbon distribution was highly right-skewed. `diameter_class` is included as a business-friendly size category for interpretation.

## 6. Select final modelling columns and remove leakage-risk variables

In [8]:
(
    model_df,
    numeric_features,
    categorical_features,
    id_columns,
    target_columns,
    dropped_leakage_columns
) = select_final_model_columns(df_engineered)

print("Model dataset shape:", model_df.shape)

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nID/context columns:")
print(id_columns)

print("\nTarget columns:")
print(target_columns)

print("\nDropped leakage-risk columns:")
print(dropped_leakage_columns)

Leakage-risk columns excluded: 8
Selected modelling dataset shape: (4540549, 23)
Numeric features: 10
Categorical features: 8
Model dataset shape: (4540549, 23)

Numeric features:
['DIA', 'HT', 'ACTUALHT', 'CR', 'STDAGE', 'BALIVE', 'ALSTK', 'GSSTK', 'DIA_squared', 'DIA_HT_interaction']

Categorical features:
['SPCD', 'STATECD', 'COUNTYCD', 'FORTYPCD', 'OWNGRPCD', 'SITECLCD', 'STDSZCD', 'diameter_class']

ID/context columns:
['CN', 'PLT_CN', 'CONDID', 'TREE']

Target columns:
['CARBON_AG', 'CARBON_AG_log']

Dropped leakage-risk columns:
['CARBON_AG_log', 'CARBON_BG', 'DRYBIO_AG', 'DRYBIO_BG', 'TPA_UNADJ', 'VOLBFNET', 'VOLCFNET', 'VOLCSNET']


In [9]:
leakage_terms = ["DRYBIO", "VOL", "CARBON_BG"]

leakage_still_present = [
    c for c in model_df.columns
    if any(term in c.upper() for term in leakage_terms)
]

print("Leakage-risk columns still present:", leakage_still_present)

Leakage-risk columns still present: []


Volume, biomass and non-target carbon variables are removed from the predictor set because they may directly overlap with the carbon target or introduce leakage. The final predictor set keeps standard field and context variables such as diameter, height, species, state, forest type, site class and stand age.

## 7. Missing values in selected modelling dataset

In [10]:
selected_missing = (
    model_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)

selected_missing.columns = ["variable", "missing_percentage"]

selected_missing

,variable,missing_percentage
0,ACTUALHT,26.296203
1,BALIVE,20.747579
2,HT,11.506076
3,DIA_HT_interaction,11.506076
4,CR,9.509401
5,GSSTK,3.045469
6,ALSTK,2.988097
7,STDAGE,1.701975
8,STDSZCD,0.378897
9,SITECLCD,0.152603


In [11]:
selected_missing[selected_missing["variable"].isin(numeric_features + categorical_features)]

,variable,missing_percentage
0,ACTUALHT,26.296203
1,BALIVE,20.747579
2,HT,11.506076
3,DIA_HT_interaction,11.506076
4,CR,9.509401
5,GSSTK,3.045469
6,ALSTK,2.988097
7,STDAGE,1.701975
8,STDSZCD,0.378897
9,SITECLCD,0.152603


Missing values are not imputed in this notebook because imputation should be fitted only on the training data inside each modelling pipeline. This avoids data leakage from validation or test data into the training process.

## 8. Final target validation

In [12]:
target_checks = {
    "rows": len(model_df),
    "missing_CARBON_AG": model_df["CARBON_AG"].isna().sum(),
    "negative_CARBON_AG": (model_df["CARBON_AG"] < 0).sum(),
    "zero_CARBON_AG": (model_df["CARBON_AG"] == 0).sum(),
    "min_CARBON_AG": model_df["CARBON_AG"].min(),
    "median_CARBON_AG": model_df["CARBON_AG"].median(),
    "mean_CARBON_AG": model_df["CARBON_AG"].mean(),
    "max_CARBON_AG": model_df["CARBON_AG"].max(),
}

pd.DataFrame(list(target_checks.items()), columns=["check", "value"])

,check,value
0,rows,4.540549e+06
1,missing_CARBON_AG,0.000000e+00
2,negative_CARBON_AG,0.000000e+00
3,zero_CARBON_AG,0.000000e+00
4,min_CARBON_AG,6.032000e-03
5,median_CARBON_AG,9.649500e+01
6,mean_CARBON_AG,4.948008e+02
7,max_CARBON_AG,1.804221e+05


## 9. Create train, validation and test splits

In [13]:
train_df, validation_df, test_df, split_summary = create_stratified_regression_split(
    model_df,
    target_col="CARBON_AG",
    random_state=42
)

split_summary

Created stratified train/validation/test split.
Train rows: 3,178,384
Validation rows: 681,082
Test rows: 681,083


,split,row_count,percentage,target_mean,target_median,target_min,target_max
0,full,4540549,100.000000,494.800823,96.494999,0.006032,180422.108817
1,train,3178384,69.999993,494.822274,96.494787,0.006032,180422.108817
2,validation,681082,14.999992,494.692738,96.494578,0.017175,114707.909346
3,test,681083,15.000014,494.808802,96.495480,0.023646,164746.908386


## 10. Check target distribution across splits

In [14]:
target_split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_df),
        "target_mean": train_df["CARBON_AG"].mean(),
        "target_median": train_df["CARBON_AG"].median(),
        "target_min": train_df["CARBON_AG"].min(),
        "target_max": train_df["CARBON_AG"].max()
    },
    {
        "split": "validation",
        "rows": len(validation_df),
        "target_mean": validation_df["CARBON_AG"].mean(),
        "target_median": validation_df["CARBON_AG"].median(),
        "target_min": validation_df["CARBON_AG"].min(),
        "target_max": validation_df["CARBON_AG"].max()
    },
    {
        "split": "test",
        "rows": len(test_df),
        "target_mean": test_df["CARBON_AG"].mean(),
        "target_median": test_df["CARBON_AG"].median(),
        "target_min": test_df["CARBON_AG"].min(),
        "target_max": test_df["CARBON_AG"].max()
    }
])

target_split_summary

,split,rows,target_mean,target_median,target_min,target_max
0,train,3178384,494.822274,96.494787,0.006032,180422.108817
1,validation,681082,494.692738,96.494578,0.017175,114707.909346
2,test,681083,494.808802,96.495480,0.023646,164746.908386


In [15]:
target_split_summary.to_csv(OUTPUTS_DIR / "final_target_split_summary.csv", index=False)

The target distribution is checked across the train, validation and test splits to confirm that the split has not created a strongly unbalanced modelling setup. Because the target is skewed, the split uses binned log-carbon values where possible to preserve the target distribution across splits.

## 11. Create final data dictionary

In [16]:
final_data_dictionary = create_final_data_dictionary(
    model_df=model_df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    id_columns=id_columns,
    target_columns=target_columns
)

final_data_dictionary

,variable_name,role,type,description
0,CARBON_AG,target,float64,Aboveground tree carbon target.
1,CN,identifier/context,int64,Unique tree record identifier.
2,PLT_CN,identifier/context,int64,Plot identifier used to link TREE records to P...
3,CONDID,identifier/context,int64,Condition identifier within a plot.
4,TREE,identifier/context,int64,Tree number within a plot.
5,DIA,numeric_predictor,float64,Tree diameter at breast height.
6,HT,numeric_predictor,float64,Tree height.
7,ACTUALHT,numeric_predictor,float64,Actual measured tree height when available.
8,CR,numeric_predictor,float64,Crown ratio.
9,STDAGE,numeric_predictor,float64,Stand age.


## 12. Create final feature list

In [17]:
final_feature_list = pd.DataFrame([
    {"variable": c, "feature_type": "numeric", "use_in_modelling": "Yes"} for c in numeric_features
] + [
    {"variable": c, "feature_type": "categorical", "use_in_modelling": "Yes"} for c in categorical_features
] + [
    {"variable": c, "feature_type": "identifier/context", "use_in_modelling": "No"} for c in id_columns
] + [
    {"variable": c, "feature_type": "target", "use_in_modelling": "Target only"} for c in target_columns
])

final_feature_list

,variable,feature_type,use_in_modelling
0,DIA,numeric,Yes
1,HT,numeric,Yes
2,ACTUALHT,numeric,Yes
3,CR,numeric,Yes
4,STDAGE,numeric,Yes
5,BALIVE,numeric,Yes
6,ALSTK,numeric,Yes
7,GSSTK,numeric,Yes
8,DIA_squared,numeric,Yes
9,DIA_HT_interaction,numeric,Yes


## 13. Handoff notes for modelling notebooks

In [18]:
handoff_notes = f"""
FIA Tree Carbon Storage Modelling Handoff Notes

Task type:
- Supervised regression.

Main target:
- CARBON_AG: individual-tree above-ground carbon storage.

Optional transformed target:
- CARBON_AG_log: log1p(CARBON_AG), included because the target distribution is highly right-skewed.

Recommended baseline model:
- Linear Regression.

Recommended evaluation metrics:
- MAE
- RMSE
- R²
- Actual vs predicted plot
- Residual plot

Important leakage warning:
- Do not use DRYBIO_AG, DRYBIO_BG, CARBON_BG, VOLCFNET, VOLCSNET, VOLBFNET or other biomass/volume/carbon-derived variables as predictors.
- CARBON_AG and CARBON_AG_log are target variables only.

Preprocessing reminder:
- Fit imputers, encoders and scalers on the training data only.
- Apply trained transformations to validation and test data.
- Do not fit preprocessing on the full dataset before splitting.

Suggested numeric predictors:
{numeric_features}

Suggested categorical predictors:
{categorical_features}

Files:
- data/processed/fia_train.parquet
- data/processed/fia_validation.parquet
- data/processed/fia_test.parquet
"""

handoff_path = OUTPUTS_DIR / "modelling_handoff_notes.txt"
handoff_path.write_text(handoff_notes)

print(handoff_notes)


FIA Tree Carbon Storage Modelling Handoff Notes

Task type:
- Supervised regression.

Main target:
- CARBON_AG: individual-tree above-ground carbon storage.

Optional transformed target:
- CARBON_AG_log: log1p(CARBON_AG), included because the target distribution is highly right-skewed.

Recommended baseline model:
- Linear Regression.

Recommended evaluation metrics:
- MAE
- RMSE
- R²
- Actual vs predicted plot
- Residual plot

Important leakage warning:
- Do not use DRYBIO_AG, DRYBIO_BG, CARBON_BG, VOLCFNET, VOLCSNET, VOLBFNET or other biomass/volume/carbon-derived variables as predictors.
- CARBON_AG and CARBON_AG_log are target variables only.

Preprocessing reminder:
- Fit imputers, encoders and scalers on the training data only.
- Apply trained transformations to validation and test data.
- Do not fit preprocessing on the full dataset before splitting.

Suggested numeric predictors:
['DIA', 'HT', 'ACTUALHT', 'CR', 'STDAGE', 'BALIVE', 'ALSTK', 'GSSTK', 'DIA_squared', 'DIA_HT_inter

## 14. Save final modelling datasets and documentation

In [23]:
save_preprocessing_outputs(
    model_df=model_df,
    train_df=train_df,
    validation_df=validation_df,
    test_df=test_df,
    data_dictionary=final_data_dictionary,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    id_columns=id_columns,
    target_columns=target_columns,
    dropped_leakage_columns=dropped_leakage_columns
)

# Save extra documentation outputs separately
final_feature_list.to_csv(
    OUTPUTS_DIR / "final_feature_list.csv",
    index=False
)

split_summary.to_csv(
    OUTPUTS_DIR / "final_split_summary.csv",
    index=False
)

target_split_summary.to_csv(
    OUTPUTS_DIR / "final_target_split_summary.csv",
    index=False
)

print("Saved full model dataset:", PROCESSED_DIR / "fia_model_data.parquet")
print("Saved train dataset:", PROCESSED_DIR / "fia_train.parquet")
print("Saved validation dataset:", PROCESSED_DIR / "fia_validation.parquet")
print("Saved test dataset:", PROCESSED_DIR / "fia_test.parquet")
print("Saved final feature list:", OUTPUTS_DIR / "final_feature_list.csv")
print("Saved split summary:", OUTPUTS_DIR / "final_split_summary.csv")
print("Saved target split summary:", OUTPUTS_DIR / "final_target_split_summary.csv")

Full modelling dataset saved to E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_model_data.parquet
Train split saved to E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_train.parquet
Validation split saved to E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_validation.parquet
Test split saved to E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_test.parquet
Saved full model dataset: E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_model_data.parquet
Saved train dataset: E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_train.parquet
Saved validation dataset: E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_validation.parquet
Saved test dataset: E:\BA_DV_Project\COMM074_FIA_Project\data\processed\fia_test.parquet
Saved final feature list: E:\BA_DV_Project\COMM074_FIA_Project\outputs\final_feature_list.csv
Saved split summary: E:\BA_DV_Project\COMM074_FIA_Project\outputs\final_split_summary.csv
Saved target split summary: E:\BA_DV_Project\C

## 15. Verify saved outputs

In [24]:
expected_processed_files = [
    "fia_model_data.parquet",
    "fia_train.parquet",
    "fia_validation.parquet",
    "fia_test.parquet"
]

for file in expected_processed_files:
    path = PROCESSED_DIR / file
    print(file, "FOUND" if path.exists() else "MISSING")

fia_model_data.parquet FOUND
fia_train.parquet FOUND
fia_validation.parquet FOUND
fia_test.parquet FOUND


In [25]:
train_reload = pd.read_parquet(PROCESSED_DIR / "fia_train.parquet")
validation_reload = pd.read_parquet(PROCESSED_DIR / "fia_validation.parquet")
test_reload = pd.read_parquet(PROCESSED_DIR / "fia_test.parquet")

print("Reloaded train:", train_reload.shape)
print("Reloaded validation:", validation_reload.shape)
print("Reloaded test:", test_reload.shape)

Reloaded train: (3178384, 23)
Reloaded validation: (681082, 23)
Reloaded test: (681083, 23)


## 17. Summary

This notebook created the final modelling-ready dataset for the FIA tree carbon storage prediction project. The cleaned dataset from Notebook 03 was used as the input, and features were selected based on the EDA findings from Notebook 04.

The final dataset keeps standard tree, site and geographic predictors such as diameter, height, crown ratio, species code, state code, forest type, ownership group, site class, stand age and stand size class. Simple engineered features were added, including `DIA_squared`, `DIA_HT_interaction`, `diameter_class` and `CARBON_AG_log`.

Leakage-risk variables related to biomass, volume or non-target carbon were excluded from the predictor set. The final data was split into training, validation and test sets using a 70/15/15 structure, with target-distribution-aware splitting where possible.

The output files are intended for the group’s later modelling notebooks. Encoding, scaling and imputation should be performed inside each modelling notebook using training data only, to avoid data leakage.